In [9]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:99% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
.inner_cell{font-size:20pt;}
div.text_cell_render pre code {font-size:20pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

<font color="red" size="6"><b>ch14. 웹 데이터 수집II</b></font>
# 1절. selenium을 이용한 동적 웹크롤링 문법
- https://selenium-python.readthedocs.io/
- `pip install selenium` (아나콘다 프롬프트)
    - 경고 무시 => pip install --upgrade requests (requests를 최신버전으로 upgrade)하거나,
                 conda install urllib3==1.26.18
- selenium버전 : 4.47 / requests버전 : 2.28.1 / urllib3버전 :2.7.0

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time

In [14]:
dv = webdriver.Chrome()
dv.get('http://python.org')

In [15]:
dv = webdriver.Chrome()
dv.get('http://python.org')
elem = dv.find_element(By.NAME, "q")
# By.CLASS_NAME, By.ID, By.CSS_SELECTOR, By.TAG_NAME
# a태그에서 By.LINK_TEXT, by.PARTIAL_LINK_TEXT
elem.clear()
elem.send_keys("pycon")
elem.send_keys(Keys.RETURN) #enter

In [19]:
dv = webdriver.Chrome()
dv.get('http://python.org')
elem = dv.find_element(By.NAME, "q")
elem.send_keys(Keys.CONTROL, "a") #crrl+a
elem.send_keys("pycon")
btn_elem = dv.find_element(By.CSS_SELECTOR, "button#submit") #GO버튼
btn_elem.click()

In [23]:
result_list = dv.find_elements(By.CSS_SELECTOR, "li>h3>a")
#len(result_list)
for reslt in result_list[:5]:
    print("{} - {}".format(reslt.text, reslt.get_attribute("href")))

PSF PyCon Trademark Usage Policy - https://www.python.org/psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette) - https://www.python.org/events/python-events/378/
PyCon Australia 2013 - https://www.python.org/events/python-events/57/
PyCon AU 2019 - https://www.python.org/events/python-events/776/
PyCon NL 2025 - https://www.python.org/events/python-events/2084/


In [25]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(dv.page_source, "html.parser")
result_list = soup.select("li>h3>a")
#len(result_list)
for result in result_list[:3]:
    print("{} - {}".format(result.text, result.attrs.get("href")))

PSF PyCon Trademark Usage Policy - /psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette) - /events/python-events/378/
PyCon Australia 2013 - /events/python-events/57/


In [27]:
from urllib.parse import urlparse
current_url = dv.current_url
print("현재 url :", current_url)
result_parse = urlparse(current_url)
print("url parsing 결과 :", result_parse)
domain = f"{result_parse.scheme}://{result_parse.netloc}"
domain = "{}://{}".format(result_parse.scheme, result_parse.netloc)
print("현재 domain :", domain)

현재 url : https://www.python.org/search/?q=pycon&submit=
url parsing 결과 : ParseResult(scheme='https', netloc='www.python.org', path='/search/', params='', query='q=pycon&submit=', fragment='')
현재 domain : https://www.python.org


In [28]:
soup = BeautifulSoup(dv.page_source, "html.parser")
result_list = soup.select("li>h3>a")
#len(result_list)
for result in result_list[:3]:
    print("{} - {}".format(result.text,
                           domain+result.attrs.get("href")))

PSF PyCon Trademark Usage Policy - https://www.python.org/psf/trademarks/pycon
PyCon Italia 2016 (PyCon Sette) - https://www.python.org/events/python-events/378/
PyCon Australia 2013 - https://www.python.org/events/python-events/57/


In [29]:
dv.close() #브라우저 종료

# 2절. 동적웹크롤링 예제
## 2.1 다음 뉴스 검색

In [46]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
news_list = [] # 뉴스제목과 뉴스 link들을 저장할 list
driver = webdriver.Chrome()
url = "https://www.daum.net/"
driver.get(url)
time.sleep(0.5) # 다음페이지가 다 실행될때까지 0.5초 대기

query = input("검색할 단어")
driver.find_element(By.CLASS_NAME, "tf_keyword").send_keys(query)
driver.find_element(By.CSS_SELECTOR, "button[type=submit]").click()
time.sleep(2) # 페이지 로딩될 시간동안 대기하기

#뉴스탭클릭
# driver.find_elements(By.CSS_SELECTOR, "ul.list_tab > li")[2].click()
driver.find_element(By.LINK_TEXT, "뉴스").click()

검색할 단어가을


In [47]:
bodies = driver.find_elements(By.CSS_SELECTOR, "strong.tit-g.clamp-g")
# len(bodies)
for body in bodies:
    a = body.find_element(By.TAG_NAME, "a")
    title = a.text
    link = a.get_attribute("href")
    # print(title, link)
    news_list.append([title, link])

In [48]:
page_nav = driver.find_element(By.CLASS_NAME, "inner_paging")
# page_nav.text
nex_page = page_nav.find_element(By.LINK_TEXT, "4") # a태그의 text가 2인 a태그
nex_page.click()

In [49]:
import pandas as pd
pd.DataFrame(news_list, columns=["뉴스제목","링크"]).shape

(10, 2)

## 2-2 다음뉴스 페이징 처리
- 위의 예제를 이용하여 원하는 페이지만큼 뉴스 검색 결과 받아오기

In [51]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
news_list = [] # 뉴스제목과 뉴스 link들을 저장할 list
driver = webdriver.Chrome()
url = "https://www.daum.net/"
driver.get(url)
time.sleep(0.5) # 다음페이지가 다 실행될때까지 0.5초 대기

query = input("검색할 단어")
driver.find_element(By.CLASS_NAME, "tf_keyword").send_keys(query)
driver.find_element(By.CSS_SELECTOR, "button[type=submit]").click()
time.sleep(2) # 페이지 로딩될 시간동안 대기하기

#뉴스탭클릭
# driver.find_elements(By.CSS_SELECTOR, "ul.list_tab > li")[2].click()
driver.find_element(By.LINK_TEXT, "뉴스").click()

pages = int(input("몇 페이지 크롤링 할까요?"))
for page in range(1,pages+1):
    bodies = driver.find_elements(By.CSS_SELECTOR, "strong.tit-g.clamp-g")
    for body in bodies:
        a = body.find_element(By.TAG_NAME, "a")
        title = a.text
        link = a.get_attribute("href")
        # print(title, link)
        news_list.append([title, link])
    page_nav = driver.find_element(By.CLASS_NAME, "inner_paging")
    nex_page = page_nav.find_element(By.LINK_TEXT, str(page+1))
    nex_page.click()
    time.sleep(2)
#driver.close()
news_df = pd.DataFrame(news_list, columns=["title", "link"])
display(news_df.head())
print(news_df.shape)

검색할 단어한교동
몇 페이지 크롤링 할까요?3


,title,link
0,헬로 키티와 피아노 치고 한교동과 헤엄쳐볼까…'호텔 플로리아 도쿄',http://v.daum.net/v/20260812151410604
1,"""한교동·헬로키티 굿즈까지""…세븐일레븐 '711데이' 프로모션",http://v.daum.net/v/20260630082016278
2,"[Tech & Now] 티머니, 첫 '한교동' 카드 출시…산리오 캐릭터 협업 확대",http://v.daum.net/v/20260722104444960
3,"티머니, 첫 ‘한교동’ 티머니카드 출시…GS25·세븐일레븐 한정 디자인",http://v.daum.net/v/20260722085729322
4,"와! 안우진이 모델 됐다...키움, 산리오 인기 캐릭터 '한교동'과 협업 상품 출시",http://v.daum.net/v/20260429061217283


(30, 2)


## 2-3 맞춤법 검사기
- 네이버 맞춤법 검사기를 이용

In [58]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time

In [68]:
driver = webdriver.Chrome()
driver.get('https://www.naver.com')
time.sleep(1)
elem = driver.find_element(By.ID, 'query')
elem.send_keys(Keys.CONTROL, 'a') # input이나 textarea나 다른 태그
elem.send_keys('맞춤법 검사기')
elem.send_keys(Keys.RETURN)
time.sleep(1)
textarea = driver.find_element(By.CLASS_NAME, 'txt_gray')
textarea.clear() # input이나 textarea
textarea.send_keys('안뇽하세요. 방갑습니다. 맛있는 점심시간 되세용')
btn = driver.find_element(By.CLASS_NAME, 'btn_check')
btn.click()
time.sleep(2)
result = driver.find_element(By.CSS_SELECTOR, 'p._result_text.stand_txt').text
print(result)
driver.close()

안녕하세요. 반갑습니다. 맛있는 점심시간 되세요


### 맞춤법검사전.txt파일을 맞춤법검사후.txt로 파일출력

In [75]:
# fp = open("data/ch14_맞춤법검사전.txt", "r", encoding="UTF-8")
# text = fp.read()
# fp.close()
with open("data/ch14_맞춤법검사전.txt", "r", encoding="UTF-8") as fp:
    text = fp.read()
ready_text_list = [] # 300자 기준으로 문장단위로 나눠진 text list
while len(text)>=300:
    print(len(text))
    temp = text[:300]
    last_dot_index = temp.rfind(".")
    ready_text_list.append(text[:last_dot_index+1])
    text = text[last_dot_index+1:]
ready_text_list.append(text)
print([len(read_text) for read_text in ready_text_list])

875
604
321
[271, 283, 279, 42]


In [7]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time
driver = webdriver.Chrome()
driver.get('https://www.naver.com')
time.sleep(0.5)
elem = driver.find_element(By.ID, 'query')

elem.send_keys('맞춤법 검사기')
elem.send_keys(Keys.RETURN)
time.sleep(0.5)
textarea = driver.find_element(By.CLASS_NAME, 'txt_gray')
results = '' # 맞춤법 검사 완료된 text
for idx, ready_text in enumerate(ready_text_list):
    print(f'검사중...{idx+1}/{len(ready_text_list)}')
    textarea.clear() # input이나 textarea
    textarea.send_keys(ready_text)
    btn = driver.find_element(By.CLASS_NAME, 'btn_check')
    btn.click()
    time.sleep(1)
    # result = driver.find_element(By.CSS_SELECTOR, 'p._result_text.stand_txt').text
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    result = soup.select_one('p._result_text.stand_txt').text
    results += result
    results = results.replace('.', '. ')
    
driver.close()

검사중...1/1


In [78]:
# 맞춤법 검사 결과(results)를 파일 출력
with open("data/ch14_맞춤법후.txt", "w") as fp:
    fp.write(results)

# 3절. 연습문제
- https://papago.naver.com/ 을 통해서 "data/ch14_맞춤법후.txt" 파일을 영문으로 번역하여 파일 출력

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time

In [3]:
driver = webdriver.Chrome()
driver.get("https://papago.naver.com")
time.sleep(0.5)
btn = driver.find_element(By.CLASS_NAME, "entry-popup-module-scss-module__UZIxta__close")
if btn:
    btn.click()
    print("팝업창 닫음")
else:
    print("팝업창 안뜸")
input_elem = driver.find_element(By.CLASS_NAME,
                                "text-translator-module-scss-module__CYJRkW__text-editor")
input_elem.send_keys("안녕하세요. 반갑습니다. 내일부터는 데이터베이스 정식 수업입니다.")
time.sleep(2)
result = driver.find_elements(By.CLASS_NAME,
                             "text-editor-module-scss-module__gKzuvW__dynamic-md")[1].text
print(result)

팝업창 닫음
Hello. Nice to meet you. Starting tomorrow, we have the official database class.


In [8]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time

user_input = input("번역할 문장을 입력하세요")
ready_text_list = [user_input]

# 네이버 맞춤법 검사
driver = webdriver.Chrome()
driver.get("https://www.naver.com")
time.sleep(1)
elem = driver.find_element(By.ID, 'query')
elem.send_keys("맞춤법 검사기")
elem.send_keys(Keys.RETURN)
time.sleep(1)
textarea = driver.find_element(By.CLASS_NAME, 'txt_gray')
results = ""
for idx, ready_text in enumerate(ready_text_list):
    print(f'검사중...{idx+1}/{len(ready_text_list)}')
    textarea.clear()
    textarea.send_keys(ready_text)
    btn = driver.find_element(By.CLASS_NAME, 'btn_check')
    btn.click()
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    result = soup.select_one('p._result_text.stand_txt').text
    results += result
    
driver.close()

with open("data/ch14_맞춤법후.txt", "w", encoding="utf-8") as fp:
    fp.write(results.strip())
print("맞춤법 검사한 파일 저장")

# 파파고 영어번역
driver = webdriver.Chrome()
driver.get("https://papago.naver.com")
time.sleep(0.5)
btn = driver.find_element(By.CLASS_NAME, "entry-popup-module-scss-module__UZIxta__close")
if btn:
    btn.click()
    print("팝업창 닫음")
else:
    print("팝업창 안뜸")
    
input_elem = driver.find_element(By.CLASS_NAME,
                                "text-translator-module-scss-module__CYJRkW__text-editor")
input_elem.click()
input_elem.send_keys(results.strip())
time.sleep(3)
translated_elem = driver.find_elements(By.CLASS_NAME,
                                      "text-editor-module-scss-module__gKzuvW__dynamic-md")
if len(translated_elem)>1:
    translated_result = translated_elem[1].text
else:
    translated_result = "번역결과를 가져오지 못함"
print(translated_result)
driver.close()

with open("data/ch14_자동화영어번역본.txt", "w", encoding="utf-8") as fp:
    fp.write(translated_result)
print("번역한 파일 저장")

번역할 문장을 입력하세요않이 왜 않되?
검사중...1/1
맞춤법 검사한 파일 저장
팝업창 닫음
Why won’t it work?
번역한 파일 저장
